# [실습 07] 오픈웨이트 vs 폐쇄형(API) — 하이브리드 라우팅

> **연계**: 제3부 07장(개발 플랫폼) · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-0.5B-Instruct`

**학습 목표**
- 요청 난이도·민감도에 따라 **오픈웨이트(로컬)** 와 **폐쇄형(API)** 로 나눠 보내는 라우터를 구현한다(07-3).
- (API는 과금·키가 필요하므로 여기서는 '모의 API'로 대체)

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import pipeline

# 오픈웨이트: 내 서버(Colab)에서 직접 실행
local = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
                 torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                 device_map="auto")

def open_weight_llm(prompt):
    msg = [{"role": "user", "content": prompt}]
    return local(msg, max_new_tokens=80, do_sample=False)[0]["generated_text"][-1]["content"].strip()

def proprietary_api(prompt):
    # 실제로는 OpenAI/Anthropic/Gemini API 호출. 여기서는 모의 응답.
    return f"[모의 폐쇄형 API 응답] '{prompt[:20]}...'에 대한 고품질 답변"

## 1. 라우터: 민감도·난이도로 모델 선택 (07-3 하이브리드)

In [ ]:
SENSITIVE = ["주민번호", "계좌", "비밀번호", "환자"]

def route(prompt):
    if any(w in prompt for w in SENSITIVE):
        # 민감 데이터 → 외부 전송 금지 → 오픈웨이트(온프레미스)
        return "오픈웨이트(민감데이터)", open_weight_llm(prompt)
    if len(prompt) > 40:
        # 복잡·고정확 필요 → 폐쇄형 API
        return "폐쇄형 API(고난도)", proprietary_api(prompt)
    return "오픈웨이트(일반)", open_weight_llm(prompt)

for p in ["안녕?", "환자 데이터를 요약해줘", "양자컴퓨팅의 최신 동향을 자세히 정리하고 향후 5년을 전망해줘"]:
    tag, ans = route(p)
    print(f"[{tag}] {p}\n → {ans}\n")

## 2. 정리
- **민감 데이터는 오픈웨이트(온프레미스)**, 고난도는 폐쇄형 API로 라우팅했다(07-3).
- 이 추상화 계층(게이트웨이)이 제공사 종속을 완화한다.
- **더 해보기**: `proprietary_api`를 실제 API 호출로 바꾸고(키 필요), 비용을 로깅해 보세요.